# FireSystem: 12-Month Burn Forecast with XGBoost

Trains an XGBoost classifier on the burn / no-burn training dataset made by `AppEEARS_API_Area.ipynb`, as a time-series forecast:

1. **Load** the region's training rows (`training_<Region>_<start>_<stop>.csv`) and its AppEEARS NetCDF files.
2. **Build lagged features** for every row (date *t*, pixel *p*), using only data available 12 months before *t*:

| Feature | Meaning |
|---|---|
| `burned_lag_12` … `burned_lag_24` | 1 if pixel *p* burned in month *t* − L |
| `burned_months_12_23` | months pixel *p* burned during *t* − 23 … *t* − 12 |
| `nbr_burned_12_23` | burned pixel-months in the 3 × 3 block around *p* over the same window |
| `ndvi_lag_12`, `evi_lag_12`, `veg_lag_12_age_days` | pixel *p*'s vegetation as of the date one year earlier, and its age |
| `lai_lag_24`, `lai_lag_24_age_days` | pixel *p*'s leaf area index (MCD15A3H) as of the date two years earlier, and its age |
| `elevation_m`, `slope_deg` | terrain of pixel *p* (SRTM, fixed in time) |
| `month`, `quarter`, `month_sin`, `month_cos`, `sensor_code` | calendar month of *t* and sensor, known in advance |
| `pixel_lat`, `pixel_lon` | centre of the 500 m pixel (burn rows sit on pixel centres, generated no-burn rows inside the pixel, so both are described by the centre) |

   Leaf area and terrain come from the GeoTIFFs in `EXTRA_DIR` (AppEEARS task with MCD15A3H `Lai_500m` + `FparLai_QC` and SRTMGL3 DEM); they are skipped if the folder is missing.

   The training dataset's own `ndvi`, `evi` and `veg_age_days` (as of the row's date) are **not** used: they are unknown 12 months ahead. `year` and `day` are not used either (trees cannot extrapolate years; no-burn days are random).

3. **Split chronologically** by fire season (July–June): the oldest seasons train, the next ones validate, the latest ones test.
4. **Walk-forward validation**: for each validation season, fit on every earlier season and predict that one; the parameter grid is ranked over these predictions. The test seasons are evaluated the same way with the chosen parameters, against two baselines (same month last year; monthly burn share).
5. **Forecast** the 12 months after the last burned-area month for the region's pixels, and save the model, scores and forecast.

The label mix is set by the sampling (up to 11 generated no-burn rows per burn), so predicted values are a **relative burn risk** for ranking pixels and months, not the real chance of burning.

## 1. Set Up

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils import chile_regions as cr
from src.utils import fire_datasets as fd
from src.utils import forecasting as fc

try:
    import xgboost as xgb
except Exception as error:  # macOS needs the OpenMP runtime
    raise ImportError("XGBoost could not be loaded. On macOS install the OpenMP runtime: brew install libomp") from error
print("xgboost", xgb.__version__)

## 2. Set Parameters
`REGION` must match a region already run in `AppEEARS_API_Area.ipynb` (its NetCDF files are in `data/AppEEARS/<Region>/` and its training file in `data/AppEEARS/`).

In [ ]:
# Region to forecast: the same name used in AppEEARS_API_Area.ipynb (e.g. "Maule", "Maule4", "Ñuble")
REGION = "Maule4"

In [ ]:
SLUG = cr.region_slug(REGION)
DATA_DIR = REPO_ROOT / "data" / "AppEEARS"
REGION_DIR = DATA_DIR / SLUG
# Leaf area / elevation GeoTIFFs: in the full region's folder (reduced regions such as Maule4 use their parent's)
PARENT_REGION = cr.REDUCED_REGIONS.get(REGION, (REGION, 1.0))[0]
EXTRA_DIR = DATA_DIR / cr.region_slug(PARENT_REGION) / "extra"
OUTPUT_DIR = DATA_DIR / "models" / SLUG

# Latest training file of the region (CSV or JSON)
candidates = sorted(
    p for p in DATA_DIR.glob(f"training_{SLUG}_*") if p.suffix in (".csv", ".json") and not p.name.endswith(".description.json")
)
if not candidates:
    raise FileNotFoundError(f"No training_{SLUG}_* file in {DATA_DIR}: run AppEEARS_API_Area.ipynb with REGION = {REGION!r} first")
TRAINING_FILE = candidates[-1]

BURN_LAGS = range(12, 25)        # months back for burn lags (all >= the forecast horizon)
VEG_LAG_MONTHS = 12              # vegetation as of the date one year earlier
LAI_LAG_MONTHS = 24              # leaf area index as of the date two years earlier
HORIZON = fc.forecast_horizon(BURN_LAGS, VEG_LAG_MONTHS)   # months ahead the features allow (12)
DROP_INCOMPLETE = True           # drop rows too early to have every burn lag (the first 24 months)

SEASON_START_MONTH = 7           # fire seasons July-June
VAL_SEASONS = 2                  # seasons for walk-forward tuning
TEST_SEASONS = 2                 # latest seasons, only for the final walk-forward evaluation
METRIC = "average_precision"     # tuning metric: "average_precision" (PR AUC), "roc_auc", "log_loss" or "brier"

PARAM_GRID = {
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [300],
    "min_child_weight": [1, 10],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_lambda": [1.0],
}

FORECAST_MONTHS = HORIZON        # months to forecast after the last burned-area month (at most HORIZON)
FORECAST_STRIDE = 2              # forecast every n-th pixel in each direction (1 = every 500 m pixel)
SEED = 42


def make_model(params: dict) -> xgb.XGBClassifier:
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        random_state=SEED, n_jobs=-1, **params,
    )


print(f"Region {REGION} | training file {TRAINING_FILE.name} | NetCDF folder {REGION_DIR}")
print(f"Forecast horizon: {HORIZON} months | burn lags {min(BURN_LAGS)}-{max(BURN_LAGS)} | vegetation lag {VEG_LAG_MONTHS} | leaf area lag {LAI_LAG_MONTHS}")
print(f"Leaf area / elevation folder: {EXTRA_DIR}" + ("" if EXTRA_DIR.exists() else "  (missing: those features are skipped)"))

## 3. Load the Training Data

In [ ]:
data = fc.load_training_data(TRAINING_FILE)
chile = cr.load_regions(REPO_ROOT / "data" / "boundaries" / cr.BOUNDARIES_FILE)
region = cr.named_region_geometry(chile, REGION)

print(f"{len(data):,} rows from {data['date'].min():%Y-%m-%d} to {data['date'].max():%Y-%m-%d}")
data.groupby(["sensor", "burned"]).size().unstack("burned")

## 4. Build the Lagged Features
#### `fc.build_forecast_features` reads each sensor's burned-area grids into a monthly burn history per pixel (cached in `histories`) and looks up the lags of every row; vegetation is sampled at the row's date minus `VEG_LAG_MONTHS`, with the same leakage-free matching as the training dataset (last valid composite ending before that date).

In [ ]:
histories = {}
features = fc.build_forecast_features(
    data, REGION_DIR, lags=BURN_LAGS, veg_lag_months=VEG_LAG_MONTHS, region=region, histories=histories,
    extra_dir=EXTRA_DIR if EXTRA_DIR.exists() else None, lai_lag_months=LAI_LAG_MONTHS,
)
FEATURES = fc.feature_columns(features)

if DROP_INCOMPLETE:
    first_complete = {s: h.indexes["month"].min() + pd.DateOffset(months=max(BURN_LAGS)) for s, h in histories.items()}
    complete = features["date"] >= features["sensor"].map(first_complete)
    print(f"Dropping {(~complete).sum():,} rows before the full lag history is available")
    features = features[complete]

print(f"{len(features):,} rows, {len(FEATURES)} features: {FEATURES}")
print("\nMissing share per feature:")
print(features[FEATURES].isna().mean().round(3).to_string())
features.head()

## 5. Split Chronologically
Whole fire seasons go to each split, oldest first: train → validation → test.

In [ ]:
df, split_seasons = fc.chronological_split(features, VAL_SEASONS, TEST_SEASONS, SEASON_START_MONTH)
summary = df.groupby("split", sort=False).agg(
    first_date=("date", "min"), last_date=("date", "max"), rows=("burned", "size"), burn_share=("burned", "mean"),
).loc[["train", "val", "test"]]
summary["seasons"] = [split_seasons[s] for s in summary.index]
summary

## 6. Tune with Walk-Forward Validation
For each validation season: fit on all earlier seasons, predict the season. Test seasons are not used here.

In [ ]:
train_val = df[df["split"] != "test"]
tuning = fc.tune_walk_forward(train_val, FEATURES, "burned", make_model, PARAM_GRID, split_seasons["val"], METRIC)
best_params = {k: tuning.iloc[0][k] for k in PARAM_GRID}
best_params = {k: int(v) if float(v).is_integer() and k in ("max_depth", "n_estimators") else float(v) for k, v in best_params.items()}
print("Best parameters:", best_params)
tuning

## 7. Evaluate on the Test Seasons (Walk-Forward)
Each test season is predicted by a model fitted on every earlier season (train + validation + earlier test seasons), with the chosen parameters. Baselines: **seasonal naive** (the pixel burned in the same month last year) and **climatology** (burn share of the calendar month in the training rows).

In [ ]:
models = {
    "xgboost": lambda: make_model(best_params),
    "seasonal_naive": fc.SeasonalNaive,
    "climatology": fc.Climatology,
}
test_predictions = {name: fc.walk_forward(df, FEATURES, "burned", factory, split_seasons["test"]) for name, factory in models.items()}
comparison = pd.DataFrame({name: fc.score_predictions(p).loc["all"] for name, p in test_predictions.items()}).T
comparison[["roc_auc", "average_precision", "log_loss", "brier"]] = comparison[["roc_auc", "average_precision", "log_loss", "brier"]].astype(float).round(4)
comparison

In [ ]:
# XGBoost per test season
fc.score_predictions(test_predictions["xgboost"]).astype(float).round(4)

## 8. Fit the Final Model

In [ ]:
final_model = make_model(best_params).fit(df[FEATURES], df["burned"])
importance = pd.Series(final_model.get_booster().get_score(importance_type="gain")).reindex(FEATURES).fillna(0)
(importance / importance.sum()).sort_values(ascending=False).round(4).rename("gain_share").to_frame()

## 9. Forecast the Next {HORIZON} Months
Rows for every `FORECAST_STRIDE`-th pixel of the region and each month after the last burned-area month; their features only use data up to that month.

In [ ]:
future = pd.concat([fc.future_rows(h, sensor, FORECAST_MONTHS, FORECAST_STRIDE) for sensor, h in histories.items()], ignore_index=True)
future = fc.build_forecast_features(
    future, REGION_DIR, lags=BURN_LAGS, veg_lag_months=VEG_LAG_MONTHS, region=region, histories=histories,
    extra_dir=EXTRA_DIR if EXTRA_DIR.exists() else None, lai_lag_months=LAI_LAG_MONTHS,
)
future["burn_risk"] = final_model.predict_proba(future[FEATURES])[:, 1]
future.insert(0, "region", REGION)

monthly_risk = future.groupby(["sensor", future["date"].dt.to_period("M")])["burn_risk"].agg(
    mean="mean", p90=lambda s: s.quantile(0.9), max="max", pixels="size",
).round(4)
print(f"{len(future):,} forecast rows ({future[['lat', 'lon']].drop_duplicates().shape[0]:,} pixels)")
monthly_risk

## 10. Save the Model, Scores and Forecast

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
period = f"{data['date'].min():%Y%m%d}_{data['date'].max():%Y%m%d}"
model_path = OUTPUT_DIR / f"xgboost_{SLUG}_{period}.json"
final_model.save_model(model_path)

forecast = future[["region", "sensor", "date", "lat", "lon", "burn_risk", *[c for c in FEATURES if c not in ("lat", "lon")]]].copy()
forecast.attrs["description"] = {
    "title": f"{HORIZON}-month burn risk forecast ({REGION})",
    "model": str(model_path.name),
    "training_file": TRAINING_FILE.name,
    "burn_risk": "XGBoost score (0-1); relative risk, since the training rows are 1 burn : up to 11 no-burn samples",
    "features": FEATURES,
    "horizon_months": HORIZON,
    "pixel_stride": FORECAST_STRIDE,
    "spatial_granularity": f"{500 * FORECAST_STRIDE} m (every {FORECAST_STRIDE}-th 500 m pixel)",
    "update_frequency": "Monthly",
}
saved = fd.save_dataframe(forecast, OUTPUT_DIR / f"forecast_{SLUG}_{period}", "csv")

report = {
    "region": REGION, "training_file": TRAINING_FILE.name, "features": FEATURES, "best_params": best_params,
    "split_seasons": {k: [int(s) for s in v] for k, v in split_seasons.items()},
    "test_comparison": json.loads(comparison.to_json(orient="index")),
}
report_path = OUTPUT_DIR / f"report_{SLUG}_{period}.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
tuning.to_csv(OUTPUT_DIR / f"tuning_{SLUG}_{period}.csv", index=False)
print("Saved:", model_path, *saved, report_path, OUTPUT_DIR / f"tuning_{SLUG}_{period}.csv", sep="\n  ")